# Overview

This notebook contains the data cleaning code for the datasets that require extensive cleaning.
- **Dataset 1:** Net Overseas Migration
- **Dataset 2:** Value of Residential Building Work Done

## Dataset 1: Net Overseas Migration

This dataset is formatted in a specific way which cannot be read by the read_csv() file from Pandas. So we will use a CSV Reader and some additional code logic to process this into a DataFrame.

In [83]:
import csv
import pandas as pd

# basic code to store all rows of the CSV file in a list of rows
rows = []
with open('../data/net_overseas_migration.csv') as file:
    reader = csv.reader(file)
    for i, row in enumerate(reader):
        rows.append(row)
print(rows)

[['', '', '', '', ''], ['Net Overseas Migration (1)', '', '', '', ''], ['Reference period by Direction of migration', '', '', '', ''], ['Counting: Persons', '', '', '', ''], ['', '', '', '', ''], ['Filters:', '', '', '', ''], ['Default Summation', 'Persons ((x1))', '', '', ''], ['', '', '', '', ''], ['Direction of migration (2)', '', 'Arrival', 'Departure', 'Total'], ['', 'Reference period (3)', '', '', ''], ['', '2006', '213410', '-105840', '107570'], ['', '2007', '460580', '-216550', '244030'], ['', '2008', '535970', '-220280', '315690'], ['', '2009', '478790', '-231890', '246900'], ['', '2010', '425120', '-253080', '172040'], ['', '2011', '449020', '-242780', '206240'], ['', '2012', '478350', '-238110', '240250'], ['', '2013', '478680', '-270310', '208380'], ['', '2014', '458760', '-276410', '182350'], ['', '2015', '473250', '-286520', '186730'], ['', '2016', '519650', '-275820', '243830'], ['', '2017', '531370', '-289710', '241660'], ['', '2018', '534400', '-282180', '252220'], [''

By padding each row to the same maximum length, we obtain a regular shape for our data that we can wrap in a DataFrame.

In [84]:
# pad all rows to the same maximum length
max_row_len = max(len(row) for row in rows)
rows = [row + [None] * (max_row_len - len(row)) for row in rows]
raw_data = pd.DataFrame(rows)
raw_data

,0,1,2,3,4
0,,,,,
1,Net Overseas Migration (1),,,,
2,Reference period by Direction of migration,,,,
3,Counting: Persons,,,,
4,,,,,
5,Filters:,,,,
6,Default Summation,Persons ((x1)),,,
7,,,,,
8,Direction of migration (2),,Arrival,Departure,Total
9,,Reference period (3),,,


## Steps complete
1. [x] Calculated the length of the longest row.
2. [x] Padded every other row to be the same length as the maximum by padding with None entries.
3. [x] Wrapped the rows in a DataFrame.

**The next steps will be to remove all unnecessary rows and initialise the DataFrame's index and rows to the correct information.**

In [85]:
# take a copy of the raw data
cleaned_data = raw_data.copy()

# We note from the peek the following:
# [] The header and footer information must be removed. This means all rows before 8 and after 31 are to be removed.
# [] The "Direction of migration (2)" column is not needed and the index should become the reference period.

# get rid of the header and footer
cleaned_data = cleaned_data.loc[8:30, :]

# drop the "Direction of migration (2)" column
cleaned_data = cleaned_data.iloc[:, 1:]

# make the direction of migration values as the column row with Arrival, Departure and Total. Drop the original row
cleaned_data.columns = cleaned_data.iloc[0]
cleaned_data = cleaned_data.loc[9:]

# rename the index to "Reference period"
cleaned_data = cleaned_data.rename_axis('Reference period', axis='rows')

# rename the columns to 'Direction of migration'
cleaned_data = cleaned_data.rename_axis('Direction of migration', axis='columns')

# drop the original row containing reference period and make the reference period be the first column. Drop that column too
cleaned_data = cleaned_data.iloc[1:, :]
cleaned_data.index = cleaned_data.iloc[:, 0]
cleaned_data = cleaned_data.iloc[:, 1:]
cleaned_data = cleaned_data.rename(columns={'Total': 'Net Arrival'})
cleaned_data

Direction of migration,Arrival,Departure,Net Arrival
,,,
2006,213410,-105840,107570
2007,460580,-216550,244030
2008,535970,-220280,315690
2009,478790,-231890,246900
2010,425120,-253080,172040
2011,449020,-242780,206240
2012,478350,-238110,240250
2013,478680,-270310,208380
2014,458760,-276410,182350


## Steps complete
1. [x] Dropped the rows that contain header and footer information.
2. [x] Made the columns equal to the Direction of Migration values.
3. [x] Renamed the column axis to "Direction of migration".
4. [x] Renamed the index to "Reference period".
5. [x] Dropped all other rows and columns until only the numerical data is left.
6. [x] Renamed 'Total' column to a more appropriate name.

In [86]:
# save the newly cleaned migration data for future use
cleaned_data.to_csv('../data/cleaned_migration_data.csv', index=True)